# Track 3: Sleep Onset Prediction
Run all cells on a Kaggle or Colab GPU. This notebook trains one multi-scale CNN + causal GRU and creates `submission.zip`.

In [ ]:
from pathlib import Path
import os, sys

ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules

# EDIT PROJECT_DIR if the notebook is not beside train.py.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'train.py').exists() and ON_KAGGLE:
    matches = list(Path('/kaggle/input').glob('**/train.py'))
    if matches:
        PROJECT_DIR = matches[0].parent

WORK_DIR = Path('/kaggle/working' if ON_KAGGLE else '/content' if ON_COLAB else PROJECT_DIR)
DATA_DIR = WORK_DIR / 'neural_compet'
OUTPUT_DIR = WORK_DIR / 'checkpoints'
assert (PROJECT_DIR / 'train.py').exists(), 'Set PROJECT_DIR to the folder containing train.py'
print({'project': str(PROJECT_DIR), 'data': str(DATA_DIR), 'output': str(OUTPUT_DIR)})

In [ ]:
# The official public pipeline is pinned to NeuralBench 0.4.0.
import subprocess
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')],
    check=True,
)

## CPU/GPU contract smoke test
This test uses synthetic EEG only. It does not produce or claim a validation score.

In [ ]:
import copy, torch
sys.path.insert(0, str(PROJECT_DIR))
from model import SleepOnsetNet
from train import weighted_l1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SleepOnsetNet().to(device)
x = torch.randn(2, 12, 4, 640, device=device)
y = torch.linspace(600, 0, 12, device=device).repeat(2, 1)
valid = torch.ones_like(y, dtype=torch.bool)
prediction, _ = model(x)
loss = weighted_l1(prediction, y, valid)
loss.backward()
model.eval(); model.reset_state()
p1 = model.predict(x[:1, 0]); p2 = model.predict(x[:1, 1])
clone = copy.deepcopy(model); clone.reset_state(); p_reset = clone.predict(x[:1, 0])
assert p1.shape == p2.shape == p_reset.shape == (1,)
assert torch.isfinite(torch.stack([p1, p2, p_reset])).all()
assert torch.allclose(p1, p_reset, atol=1e-5), 'reset_state is not reproducible'
print(f'Smoke test passed on {device}; parameters={sum(p.numel() for p in model.parameters()):,}')

## Download, prepare, train, validate, and package
The first run downloads about 1.1 GB and creates the official NeuralBench cache. Kaggle Internet must be enabled.

In [ ]:
import subprocess
command = [
    sys.executable, str(PROJECT_DIR / 'train.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--download',
    '--epochs', '30',
    '--batch-size', '16',
    '--learning-rate', '0.001',
    '--sequence-length', '12',
    '--gru-hidden-size', '128',
    '--patience', '7',
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
import zipfile
submission_zip = OUTPUT_DIR.parent / 'submission.zip'
with zipfile.ZipFile(submission_zip) as archive:
    names = archive.namelist()
assert sorted(names) == ['best_model.pt', 'submission.py']
assert (OUTPUT_DIR / 'best_model.pt').is_file()
print('Best checkpoint:', OUTPUT_DIR / 'best_model.pt')
print('Upload to Codabench:', submission_zip)
print('ZIP root:', names)